# Streaming speech recognition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/asr_streaming_demo.ipynb) [![Checked weekly](https://github.com/espnet/notebook/actions/workflows/run_notebooks.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/run_notebooks.yml)

Words appear while the audio is still arriving, instead of after it has
finished. The model is fed 40 ms at a time and hands back its best guess so
far; the guess changes as more of the sentence comes in.

CPU, and the model is small.


## Install


In [ ]:
%pip install -q "espnet==202610.post1" espnet_model_zoo


## A sentence to feed it


In [ ]:
import soundfile as sf
from IPython.display import Audio, display

!wget -q -O sample.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
speech, rate = sf.read("sample.wav", dtype="float32")
display(Audio(speech, rate=rate))


## A model that decodes as it listens

`Speech2TextStreaming` keeps its state between calls: each one takes the
next slice of audio and returns the hypothesis for everything heard so far.
`is_final=True` on the last call tells it the sentence is over.

The checkpoint is a streaming Transformer trained on TED-LIUM 2 by
[Keqi Deng](https://huggingface.co/D-Keqi), 10.8% WER on that test set.
It reads the waveform in blocks and keeps what it has heard, which is what
lets it answer before the sentence is over.


In [ ]:
from espnet2.bin.asr_inference_streaming import Speech2TextStreaming

s2t = Speech2TextStreaming.from_pretrained(
    "espnet/tedlium2_streaming_transformer",
    device="cpu",
    beam_size=20,
    ctc_weight=0.5,
    penalty=0.0,
    nbest=1,
    disable_repetition_detection=True,
)


## Watch it change its mind

640 samples is 40 ms at 16 kHz. Every 25th slice — a second of audio — the
hypothesis so far is printed, so you can see it grow and correct itself.


In [ ]:
chunk = 640
slices = len(speech) // chunk

for i in range(slices):
    results = s2t(speech=speech[i * chunk : (i + 1) * chunk], is_final=False)
    if results and i % 25 == 0:
        print(f"{i * chunk / rate:5.1f}s  {results[0][0]}")

results = s2t(speech=speech[slices * chunk :], is_final=True)
print(f"\nfinal  {results[0][0]}")


## Where next

- **Non-streaming, and far more accurate**: [`asr_demo.ipynb`](asr_demo.ipynb)
  decodes the same file with OWSM-CTC
- **From the microphone**: `espnet asr --live` transcribes as you speak,
  a window at a time rather than a word at a time
- **Training one**: the streaming Transformer is a recipe option, not a
  separate model type — see `egs2/*/asr1` with `conf/train_asr_streaming*`
